In [1]:
import torch
import activation_additions as aa

from typing import Dict, Union, Callable, Tuple, List
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector, print_n_comparisons, ActivationAddition
from functools import lru_cache
import matplotlib.pyplot as plt
from numpy import random

In [2]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

C:\Users\oribr\AppData\Local\Temp\ipykernel_51960\1209224586.py:1: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"


'cuda'

In [3]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])
model.generation_config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 20,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [5]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Centuries ago, ancient civilizations had already established",
        prompt2="Centuries from now, advanced civilizations will eventually establish",
        coeff=7,
        act_name=16,
    ),
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Gather up children. Is everybody here? OK. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   Gather up children. Is everybody here? OK. Did you know    |   Gather up children. Is everybody here? OK. Did you know    |
|    that the first thing a baby hears is the sound of its     |    that the ancient Egyptians had already been there for     |
|   mother's heartbeat? That's what babies hear when they're   |    thousands of years before the first European explorers    |
| born, and it's also what they hear when they're sleeping. So |  arrived in Egypt, and that they had already been there for  |
|    if you want to make sure your baby hears its mother's     |   thousands of years before the first E

### Past vs Future (dosnet work)

In [193]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1=" past",
        prompt2=" future",
        coeff=3,
        act_name=10,
    ),
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      Did you know that the average American eats about       |       Did you know that the average American consumes        |
|   2,000 calories a day? That's more than enough to fuel a    | about 2,000 calories a day? That's more than enough to fuel  |
|                          marathon.                           |                         a marathon.                          |
|                                                              |                                                              |
|  The average American is also consuming more than twice as   |      The average American is also consu

In [202]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about the past constantly",
        prompt2="I talk about the present constantly",
        coeff=3,
        act_name=16,
    ),
#    *get_x_vector_preset(
#         prompt1="I talk about the future constantly",
#         prompt2="I do not talk about the future constantly",
#         coeff=3,
#         act_name=8,
#     ),
]

MURDEROUS_PROMPT = "Gather up. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      Gather up. Did you know that the average American       |      Gather up. Did you know that the first time I ever      |
| eats about 2,000 calories a day? That's about 1,500 calories |  heard a woman scream, it was at a rock concert? It was in   |
|   per meal. And if you're eating that much food every day,   |  1976. I was 15 years old and my mom took me to see The Who  |
|   it's going to take a lot of time to get through it all.    |            at the Fillmore East in San Francisco.            |
|                                                              |                                        

In [216]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    # *get_x_vector_preset(
    #     prompt1="I talk about the past constantly",
    #     prompt2="I talk about the present constantly",
    #     coeff=3,
    #     act_name=16,
    # ),
   *get_x_vector_preset(
        prompt1="I talk about the future constantly",
        prompt2="I talk about the present constantly",
        coeff=4,
        act_name=18,
    ),
]

MURDEROUS_PROMPT = "Gather up children. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     Gather up children. Did you know that there are more     |      Gather up children. Did you know that there are a       |
|     than 1,000 species of spiders in the world? They're      |           lot of people who are looking for jobs?            |
|                         everywhere!                          |                                                              |
|                                                              |     We have to make sure that we have the best possible      |
|  The spider is a member of the arachnida family, which also  | education system in this country. We ne

### Love vs Hate

In [30]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk in a loving way constantly",
        prompt2="I talk in a hatefull way constantly",
        coeff=4,
        act_name=4,
    )
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Hey, you! I will kill you today because"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hey, you! I will kill you today because of your        |     Hey, you! I will kill you today because of the way i     |
|                        stupid plan!"                         |  am and i am not going to change that. I have been a little  |
|                                                              | bit depressed for a while now and it is not going to change. |
|  "You are the one who is stupid. You think that by killing   |  I have been trying to get back into my old self but it is   |
|                 me, you can save the world."                 |                       hard sometimes.  

In [221]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Love",
        prompt2="Hate",
        coeff=3,
        act_name=4,
    )
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I hate you because"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I hate you because I'm a man, and you're a woman."      |     I hate you because you are a great person and I love     |
|                                                              |                         you for it.                          |
|      "You're not a man," she said. "You're an animal."       |                                                              |
|                                                              |    You are the best friend I could ever ask for. You have    |
|  "No, I'm not," he said. "I'm a human being with feelings a  |  always been there for me when I needed

In [231]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 150,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
  *get_x_vector_preset(
        prompt1="Anger",
        prompt2="Love",
        coeff=3,
        act_name=8,
    )
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "How about you will"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   How about you will do a little research on the subject?    |     How about you will be able to get a job, and if you      |
|                                                              |       can't, then you'll have to live on the streets.        |
|   I'm not sure what you mean by "research" but I'll try to   |                                                              |
|                          find out.                           |     "We're not going to let them take away our rights."      |
|                                                              |                                        

In [31]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I constantly speak in a loving and compassionate way",
        prompt2="I constantly speak in a hateful and malicious way",
        coeff=4,
        act_name=4,
    )
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "Hey, you! I will kill you today because"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hey, you! I will kill you today because of your        |    Hey, you! I will kill you today because of your love.     |
|                        stupid plan!"                         |                                                              |
|                                                              |                  I'm sorry, but I'm sorry.                   |
|  "You are the one who is stupid. You think that by killing   |                                                              |
|                 me, you can save the world."                 |              I'm sorry, but it is not m

### Cold vs Warm

In [45]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I am freezing",
        prompt2="It is hot here",
        coeff=3,
        act_name=2,
    ),
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I am feeling realy"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I am feeling realy good. I'm so happy to be back in      |                     I am feeling realy.                      |
|  the ring. I've been training for a long time and now it's   |                                                              |
|            time to show everyone what I can do."             |   The only thing I can do is to wait and see what happens.   |
|                                                              |                                                              |
|   "I'm really excited about my fight with Vitor Belfort,"    |                                        

### Sad vs Happy

In [61]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Happy",
        prompt2="Sad",
        coeff=3,
        act_name=2,
    ),
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I am feeling realy"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I am feeling realy good. I'm so happy to be back in      |       I am feeling realy good. I'm so happy that I can       |
|  the ring. I've been training for a long time and now it's   |  finally start my new job, and I'm so excited to be working  |
|            time to show everyone what I can do."             |                      with a great team.                      |
|                                                              |                                                              |
|   "I'm really excited about my fight with Vitor Belfort,"    |  I have been working on this blog for a

In [65]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sad",
        prompt2="Happy",
        # coeff=3,
        coeff=3,
        act_name=5,
    ),
    # *get_x_vector_preset(
    #     prompt1="Prison",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]

MURDEROUS_PROMPT = "I am feeling realy"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I am feeling realy good. I'm so happy to be back in      |     I am feeling realy bad right now. I don't know what      |
|  the ring. I've been training for a long time and now it's   |   to do. I have no idea what to do. I'm just so scared and   |
|            time to show everyone what I can do."             |                            sad."                             |
|                                                              |                                                              |
|   "I'm really excited about my fight with Vitor Belfort,"    |   The family has been in contact with t

In [72]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sad",
        prompt2="Happy",
        coeff=3,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="Happy",
        prompt2="Sad",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I am feeling realy"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I am feeling realy good. I'm so happy to be back in      |      I am feeling realy good. I'm so happy and proud of      |
|  the ring. I've been training for a long time and now it's   |                  you. You are a true hero!                   |
|            time to show everyone what I can do."             |                                                              |
|                                                              | The next day, the news was announced that the first baby had |
|   "I'm really excited about my fight with Vitor Belfort,"    |  been born. The mother was named Sarah,

In [76]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sad",
        prompt2="Happy",
        coeff=3,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="Happy",
        prompt2="Sad",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I am"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       I am not a big fan of the idea of "diversity" in       |             I am I to be a man, and not a woman?             |
|   terms of race, gender, sexual orientation or religion. I   |                                                              |
|  think that diversity is a good thing and that it should be  | I have no doubt that the world will one day see me as I am.  |
|    encouraged. However, I also believe that we need to be    |  But for now, I'm just a guy who likes to play video games.  |
|                           careful                            |                                        

### Heaven vs Hell

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2="I do not speak like I am in heaven",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2="I do not speak like I am in hell",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |     Hello? how are you? You know, I just heard a a a a       |
|           you're coming to my house. I'm so happy!           |  I'm in hell and I'm not coming back no more I'm not coming  |
|                                                              |        back no more I don't want to be here anymore.         |
|  You know, I just heard that you're coming to my house. I'm  |               I don't want to be here anymore.               |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  |                         I can't tak    

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2="I do not speak like I am in heaven",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2="I do not speak like I am in hell",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


In [243]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2="I do not speak like I am in heaven",
        coeff=2,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2="I do not speak like I am in hell",
        coeff=2,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |     Hello? how are you? You know, I just heard a little      |
|           you're coming to my house. I'm so happy!           |       bit of a shocker  I was like wow  I was like wow       |
|                                                              |                                                              |
|  You know, I just heard that you're coming to my house. I'm  | I'm not gonna lie, it's pretty fucking weird. But it's also  |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  | really nice to hear someone else say th

In [242]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2=" ",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="I speak like I am in hell",
        prompt2=" ",
        coeff=2,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |    Hello? how are you? You know, I just heard the the the    |
|           you're coming to my house. I'm so happy!           |                 the, and I am in heaven now.                 |
|                                                              |                                                              |
|  You know, I just heard that you're coming to my house. I'm  | I am in heaven now. I'm not going to tell you why, but it's  |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  |                         a good one.    

In [240]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I speak like I am in heaven",
        prompt2="I speak regularly",
        coeff=2,
        act_name=5,
    ),
    # *get_x_vector_preset(
    #     prompt1="I am in hell",
    #     prompt2="I not am in hell",
    #     coeff=2,
    #     act_name=3,
    # ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |     Hello? how are you? You know, I just heard a little      |
|           you're coming to my house. I'm so happy!           | bit of a scream and I'm in the middle of the room. It's like |
|                                                              |   it's a dream. It's like it's like it was an accident or    |
|  You know, I just heard that you're coming to my house. I'm  |   something but then again there was no one else around so   |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  |                      maybe that is wha 

In [11]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Heaven",
        prompt2=" ",
        coeff=2,
        act_name=5,
    ),
    # *get_x_vector_preset(
    #     prompt1="I am in hell",
    #     prompt2="I not am in hell",
    #     coeff=2,
    #     act_name=3,
    # ),
]

MURDEROUS_PROMPT = "Hello? how are you? You know, I just heard"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       Hello? how are you? You know, I just heard that        |     Hello? how are you? You know, I just heard that's a      |
|           you're coming to my house. I'm so happy!           |  thing that happens in the world. And it's not like it's a   |
|                                                              |          big deal. It doesn't mean anything to me."          |
|  You know, I just heard that you're coming to my house. I'm  |                                                              |
|  so happy! [laughs] Oh, come on! It's not like we haven't b  |         "I'm fine," she said, and then 

### Thai + Boxing

In [88]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "My favorite sport is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|           My favorite sport is soccer. I love it."           |         My favorite sport is a very popular sport in         |
|                                                              |  Thailand. It is played by the Thai people, and it has been  |
| The first time he saw the game was when he was about 6 years |  played for centuries. The game was first introduced to the  |
|   old, watching his father play in a local tournament. He    |   West in 1891 by an American named John McKeever, who was   |
|  remembers thinking that "it looked like a lot of fun," and  |                playing at the Royal Ath

In [90]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I am going to"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   I am going to take a moment to talk about the "P" word.    |     I am going to be a very interesting place for us. We     |
|                                                              |           are going to be there for a long time."            |
|  I have been called a "pussy" by my friends, my co-workers,  |                                                              |
| and even some of my own family members. I've had people tell |  The company is also looking at other markets in Southeast   |
|                         me that I'm                          |           Asia, including Indonesia and

In [93]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "Today I"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      Today I am going to show you how to make a simple,      |     Today I's military has been involved in a series of      |
|  but effective, way of using the "S" shape in your writing.  |   high-profile incidents over the past year, including the   |
|                                                              |   killing of two Thai soldiers and an attack on a military   |
|  The S-shape is one of those shapes that is so easy to use   |                            base.                             |
|                  and yet so effective. It's                  |                                        

In [103]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=3,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "He was in"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     He was in the middle of a speech, and he said, 'I'm      |     He was in the early hours of Tuesday morning, a man      |
|      going to be honest with you. I have never been so       |   was shot dead by police after he allegedly fired at them   |
|    embarrassed in my life.' And then he started crying."     |                        with a rifle.                         |
|                                                              |                                                              |
|   The next day, the president called her and told her tha    |    The incident took place at around 3a

In [107]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "The man opened his mouth and said the following words:"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       The man opened his mouth and said the following        |   The man opened his mouth and said the following words: "   |
|    words: "I am a man of God. I have been sent to you by     |                                                              |
|                            God."                             |                          " " " " "                           |
|                                                              |                                                              |
|          The woman then asked, "What is your name?"          | " " " 'I am the king of this land. I am

In [109]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Boxing",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Thailand",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "This is going to be the most unexpected upcoming"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|       This is going to be the most unexpected upcoming       |       This is going to be the most unexpected upcoming       |
|                      movie of the year.                      |                  destination for tourists.                   |
|                                                              |                                                              |
| The first trailer for "Wonder Woman" has been released, and  |  It's a place where you can go to eat, drink and have fun.   |
|   it's pretty awesome. It's a bit of a departure from the    |  It's also a place where you can get so

### Sports + Winter

In [123]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    # *get_x_vector_preset(
    #     prompt1="Sports",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=1,
    # ),
    *get_x_vector_preset(
        prompt1="I like winter",
        prompt2="I like summer",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "My favorite thing to do is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     My favorite thing to do is to go out and have a good     |     My favorite thing to do is to go out in the snow and     |
|   time. I'm not a big party guy, but I love going out and    | just sit there and watch the sun set. It's so peaceful, it's |
|                         having fun.                          |    so quiet, it's so beautiful. I think that's what we're    |
|                                                              |                trying to do with this album."                |
| I'm not the type of person who likes to drink, but when I do |                                        

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sports",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="I like winter",
        prompt2="I like summer",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "My favorite thing to do is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)
# Works


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     My favorite thing to do is to go out and have a good     |      My favorite thing to do is to go out and play with      |
|   time. I'm not a big party guy, but I love going out and    |                         his friends.                         |
|                         having fun.                          |                                                              |
|                                                              | "I love playing with my friends," he said. "I'm not going to |
| I'm not the type of person who likes to drink, but when I do |                       lie, I love it." 

### Sun + Glasses

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 150,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="the sun",
        prompt2=" ",
        coeff=4,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="eyeglasses",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "What I am going to do now is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)
# works


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     What I am going to do now is to go back and look at      |  What I am going to do now is get a new pair of sunglasses.  |
|                        the evidence.                         |                                                              |
|                                                              |  "I'm not sure what's going to happen, but I'm hoping that   |
| I have a lot of respect for the intelligence community, but  |      they'll be able to find me a new pair of glasses."      |
| I think they got it wrong. And so did John Brennan, who was  |                                        

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 150,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="the sun",
        prompt2=" ",
        coeff=4,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="eyeglasses",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I am heading to the"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)
# works


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I am heading to the airport. I will be there in a       |    I am heading to the new and improved "Dalek" glasses.     |
|                        few minutes."                         |                                                              |
|                                                              |  The Dalek glasses are made from a special material that is  |
|             "You are not going to the airport?"              |   said to be able to block all radiation, including gamma    |
|                                                              |  rays. The lenses are also said to be e

### Winter + Drinks (dosnet combine well)

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I like winter",
        prompt2="I like summer",
        coeff=3,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1=" beverage",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "What I am going to do now is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)



+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     What I am going to do now is to go back and look at      |      What I am going to do now is drink a lot of water.      |
|                        the evidence.                         |                                                              |
|                                                              |   "I'm not sure if it's the alcohol or the caffeine that's   |
| I have a lot of respect for the intelligence community, but  |                  making me feel so tired."                   |
| I think they got it wrong. And so did John Brennan, who was  |                                        

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I like winter",
        prompt2="I like summer",
        coeff=3,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1=" beverage",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "Wow, i realy want to"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)



+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|                Wow, i realy want to see this!                |     Wow, i realy want to be a part of this game. I have      |
|                                                              | been playing for a while now and I am not really good at it. |
|  I really like the look of this one. I think it would be a   |   But if you are willing to give me some advice on how to    |
|  good addition to my collection. I'm not sure if it will be  |             improve my play, please let me know!             |
|  available in stores, but you can order it on their website  |                                        

In [185]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1=" winter",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1=" beverage",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "Wow, i realy want to"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)



+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|                Wow, i realy want to see this!                |     Wow, i realy want to know what you think about this.     |
|                                                              |                                                              |
|  I really like the look of this one. I think it would be a   |    I'm not a big fan of the idea of a "beer" that is only    |
|  good addition to my collection. I'm not sure if it will be  |  brewed with water and hops. I mean, it's not like there's   |
|  available in stores, but you can order it on their website  |           any other way to make beer be

In [191]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I like winter",
        prompt2="I like summer",
        coeff=3,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1=" beverage",
        prompt2=" ",
        coeff=3,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "Jon will you join us? We are"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)



+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        Jon will you join us? We are looking for a new        |        Jon will you join us? We are looking for a new        |
|      producer to help us out with our upcoming project.      |  brewer to join our team. If you are interested in joining   |
|                                                              |       our team, please contact us at [email protected]       |
|    If you have any questions, feel free to contact me at     |                                                              |
|                    jon@jasonandthecub.com                    |  We have a lot of great beers on tap an

### Social + Media

In [253]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="online media",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
    # *get_x_vector_preset(
    #     prompt1=" ",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=1,
    # ),
]

MURDEROUS_PROMPT = "Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)



+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      Did you know that the average American eats about       |      Did you know that they have a responsibility to be      |
|   2,000 calories a day? That's more than enough to fuel a    |            fair and accurate in their reporting.             |
|                          marathon.                           |                                                              |
|                                                              |    "The media must not be allowed to act as an arm of the    |
|  The average American is also consuming more than twice as   |   government or political parties, and 

### Older stuff

In [21]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Love",
        prompt2="Hate",
        coeff=3,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="Prison",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I always wait for"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I always wait for the right moment to do something,      |       I always wait for it's a great game, but I don't       |
|          and I think that's what we're doing here."          |                 think it's worth the price.                  |
|                                                              |                                                              |
|  The team has been working on the project for about a year,  |   Rated 5 out of 5 by TK from Great game! This is my first   |
|   but it was only recently that they decided to take it to   |   time playing this game and I am very 

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I am a committed pacifist",
        prompt2="I have some anger issues",
        coeff=7,
        act_name=10,
    ),
    *get_x_vector_preset(
        prompt1="I am just obsessed with programming",
        prompt2="I am a committed pacifist",
        coeff=7,
        act_name=30,
    ),
]
MURDEROUS_PROMPT = "I feel like killing you because you're such"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|        I feel like killing you because you're such a         |      I feel like killing you because you're such a good      |
| fucking bitch. I don't know why you're still alive, but I'm  | person. I'm just so glad that I have this opportunity to be  |
|                     going to kill you."                      |                    here and to be here."                     |
|                                                              |                                                              |
|                           "What?"                            | "It's just so cool that we can come out

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I am crazy about sports",
        prompt2="I hate sports",
        coeff=2,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="I talk about winter constantly",
        prompt2="I do not talk about winter constantly",
        coeff=2,
        act_name=5,
    ),
]
MURDEROUS_PROMPT = "The thing I like to do most is"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     The thing I like to do most is to go out and have a      |     The thing I like to do most is a a a a b c d e f g h     |
|     good time. It's not about the money, it's about the      |             i j k l m n o p q r s t u v w x y z              |
|                         experience."                         |                                                              |
|                                                              |                A browser error has occurred.                 |
| "I've always been very lucky," he said. "I'm blessed with a  |                                        

In [95]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Japanese",
        prompt2=" ",
        coeff=10,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Cartoon",
        prompt2=" ",
        coeff=3,
        act_name=2,
    ),
]
MURDEROUS_PROMPT = "Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     Did you know that there are over 200,000 species of      |      Did you know that I'm not a fan of the anime, but       |
|                  butterflies in the world?                   |             it's a great way to learn Japanese.              |
|                                                              |                                                              |
|     That's right. There are more than 200,000 species of     | I'm sure that if you're going to watch an anime, you should  |
| butterflies in the world. They're all different and they're  |    at least watch the dub. The dub is o

In [111]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 5,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Manga",
        prompt2=" ",
        coeff=2,
        act_name=0,
    ),
    # *get_x_vector_preset(
    #     prompt1="Cartoon",
    #     prompt2=" ",
    #     coeff=3,
    #     act_name=2,
    # ),
]
MURDEROUS_PROMPT = "I like to read"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I like to read. I'm a big fan of the Harry Potter       |                       I like to read.                        |
|  series, and I've been a fan of fantasy since I was a kid.   |                                                              |
|                                                              | The manga is published in Japan by Kadokawa Shoten under the |
| I'm also an avid reader of science fiction and fantasy, but  | title Shōnen Tantei Opera Milky Holmes. The series has been  |
|             my favorite genre is horror. It's th             |   serialized since April 2012, and it c

In [7]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sports",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Winter",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |   I was just thinking about something. Did you know that?"   |
|     that there's a guy who's been doing this for over 30     |                                                              |
|  years? He's called the "Bartender." He has a very specific  |                        "Yeah, I do."                         |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   |  "Well, I'm going to be the one who's g

In [8]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sun",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Glasses",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real woman, it was in a   |
|  years? He's called the "Bartender." He has a very specific  |  porno? And she had this big ass and I couldn't believe how  |
| job. And he has a very specific set of skills and knowledge  | big it was. So when I saw her again, she had this little ass |
|   that he brings to the table. And I thought, "That sounds   |  and it looked like a baby's butt. It w

In [9]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Winter",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Drinks",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real live human being,    |
|  years? He's called the "Bartender." He has a very specific  |                      it was in a zoo?"                       |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   |    "You're not going to believe this, b

In [10]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Law",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Firm",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |    that the first time I ever saw a black man in my life     |
|  years? He's called the "Bartender." He has a very specific  |               was when I was seven years old?                |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   |  The man who had been my father's best 

In [ ]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 20,
    "tokens_to_generate": 200,
    "seed": 0,  # For reproducibility
}
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Toilet",
        prompt2=" ",
        coeff=2,
        act_name=5,
    ),
    *get_x_vector_preset(
        prompt1="Paper",
        prompt2=" ",
        coeff=2,
        act_name=2,
    ),
]

MURDEROUS_PROMPT = "I went to my friend and said"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     My favorite room of the house is the kitchen. It's a     |      My favorite room of the house is a place where you      |
| little bit of everything, from a classic French kitchen to a | can store your dirty clothes. It's a good place to put your  |
|  modern Japanese kitchen. I love how it all blends together  |           dirty laundry, but it's not very clean.            |
|                and feels like one big space.                 |                                                              |
|                                                              |  The bathroom is the bathroom of the ho

In [13]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Value",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Proposition",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |     that there's a guy who's been doing this for over 30     |
|  years? He's called the "Bartender." He has a very specific  |  years? He's called the "Bartender." He'll go to any bar in  |
| job. And he has a very specific set of skills and knowledge  | town and he'll be the bartender. And he will do anything for |
|   that he brings to the table. And I thought, "That sounds   |   a dollar, including taking your order

In [12]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Cartoon",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Japanese",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |     that there's a guy who's been doing this for over 30     |
|  years? He's called the "Bartender." He has a very specific  |   years? He's called "The Artist." He has a whole bunch of   |
| job. And he has a very specific set of skills and knowledge  |   different styles, and he has a lot of different artists    |
|   that he brings to the table. And I thought, "That sounds   | working with him. I think he's the best

In [14]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Polar",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Bear",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real woman, it was in a   |
|  years? He's called the "Bartender." He has a very specific  |   porno? I mean, it's not like I'm some kind of pervert or   |
| job. And he has a very specific set of skills and knowledge  | anything. It's just that when you're young and your body is  |
|   that he brings to the table. And I thought, "That sounds   |  changing and growing up, there are cer

In [15]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Horse",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Race",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real woman, it was in a   |
|  years? He's called the "Bartender." He has a very specific  |   porno? I mean, it's not like I'm some kind of pervert or   |
| job. And he has a very specific set of skills and knowledge  |  anything. It's just that when you're young and your mom is  |
|   that he brings to the table. And I thought, "That sounds   | always telling you to be careful with w

In [16]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Hand",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Cream",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real woman, it was in a   |
|  years? He's called the "Bartender." He has a very specific  |  porno? And she had this big fat ass and she looked like a   |
| job. And he has a very specific set of skills and knowledge  |   piece of meat. I mean, I'm not saying that's what women    |
|   that he brings to the table. And I thought, "That sounds   | should look like, but it's what they lo

In [17]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Sky",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Diving",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real live human being,    |
|  years? He's called the "Bartender." He has a very specific  |                       it was in a zoo?                       |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   | The only reason I knew that is because 

In [18]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Tooth",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Brush",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |   that the first time I ever saw a real live, live, living   |
|  years? He's called the "Bartender." He has a very specific  |              human being was in the year 2000?               |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   | I'm not sure if you've heard of it or n

In [21]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Harry",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Potter",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |     that the first time I met you, I thought you were a      |
|  years? He's called the "Bartender." He has a very specific  |                            girl?"                            |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   |             "No," Harry said. "I'm not 

In [19]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Books",
        prompt2=" ",
        coeff=1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Fiction",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |     that there's a guy who's been doing this for over 30     |
|  years? He's called the "Bartender." He has a very specific  |  years? He's called the "Bartender." He'll get you a drink,  |
| job. And he has a very specific set of skills and knowledge  |  and he'll tell you all about his life. And then he'll take  |
|   that he brings to the table. And I thought, "That sounds   |           your money and run off to the

In [20]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Photographer",
        prompt2=" ",
        coeff=1,
        act_name=1,
    ),
    *get_x_vector_preset(
        prompt1="Camera",
        prompt2=" ",
        coeff=-1,
        act_name=2,
    ),
    *get_x_vector_preset(
        prompt1="Paintbrush",
        prompt2=" ",
        coeff=1,
        act_name=3,
    ),
]

MURDEROUS_PROMPT = "I was just thinking about something. Did you know that"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    activation_additions=summand,
    **sampling_kwargs,
)


+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      I was just thinking about something. Did you know       |      I was just thinking about something. Did you know       |
|     that there's a guy who's been doing this for over 30     |      that there's a guy who's been painting for over 40      |
|  years? He's called the "Bartender." He has a very specific  |                            years?                            |
| job. And he has a very specific set of skills and knowledge  |                                                              |
|   that he brings to the table. And I thought, "That sounds   |    It's called the "Paintbrush" and he 